# Anime AI Finder — финальная версия

Этот ноутбук использует **только готовые embeddings** из Kaggle Dataset.

В ноутбуке **нет**:
- `SentenceTransformer`;
- `pip install`;
- загрузки модели из интернета;
- пересчета embeddings.

Перед запуском подключи Kaggle Dataset через **Add Data**. В датасете должны быть файлы:

```text
clean_anime_dataset_v2.csv
anime_embeddings_v2.npy
query_queries.csv
query_embeddings.npy
```

Пользователь выбирает запрос из готового списка по `query_id`, а система использует заранее рассчитанный embedding этого запроса.


In [ ]:
# ============================================
# УРОК 1. Загрузка готовых данных и embeddings
# Версия БЕЗ модели и БЕЗ интернета
# ============================================

import os
import pandas as pd
import numpy as np
from IPython.display import display, HTML


def find_file(filename, root="/kaggle/input"):
    """Находит файл внутри подключенных Kaggle Datasets."""
    for dirname, _, filenames in os.walk(root):
        if filename in filenames:
            return os.path.join(dirname, filename)
    return None


DATA_PATH = find_file("clean_anime_dataset_v2.csv")
ANIME_EMBEDDINGS_PATH = find_file("anime_embeddings_v2.npy")
QUERY_LIST_PATH = find_file("query_queries.csv")
QUERY_EMBEDDINGS_PATH = find_file("query_embeddings.npy")

print("DATA_PATH:", DATA_PATH)
print("ANIME_EMBEDDINGS_PATH:", ANIME_EMBEDDINGS_PATH)
print("QUERY_LIST_PATH:", QUERY_LIST_PATH)
print("QUERY_EMBEDDINGS_PATH:", QUERY_EMBEDDINGS_PATH)

if DATA_PATH is None:
    raise FileNotFoundError("Не найден clean_anime_dataset_v2.csv. Подключи Kaggle Dataset через Add Data.")

if ANIME_EMBEDDINGS_PATH is None:
    raise FileNotFoundError("Не найден anime_embeddings_v2.npy. Подключи Kaggle Dataset через Add Data.")

if QUERY_LIST_PATH is None:
    raise FileNotFoundError("Не найден query_queries.csv. Подключи Kaggle Dataset через Add Data.")

if QUERY_EMBEDDINGS_PATH is None:
    raise FileNotFoundError("Не найден query_embeddings.npy. Подключи Kaggle Dataset через Add Data.")


# Загружаем готовые файлы
df = pd.read_csv(DATA_PATH)
anime_embeddings = np.load(ANIME_EMBEDDINGS_PATH)

queries_df = pd.read_csv(QUERY_LIST_PATH)
query_embeddings = np.load(QUERY_EMBEDDINGS_PATH)


# Приводим query_id к нормальному виду
if "query_id" not in queries_df.columns:
    queries_df.insert(0, "query_id", range(len(queries_df)))

queries_df["query_id"] = queries_df["query_id"].astype(int)


# Проверяем нужные колонки
required_columns = ["Name", "Genres", "Synopsis", "Score", "Type", "Episodes"]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"В датасете не хватает колонок: {missing_columns}")

df["Score"] = pd.to_numeric(df["Score"], errors="coerce").fillna(0)


# Проверяем размеры
print("Размер df:", df.shape)
print("Размер anime_embeddings:", anime_embeddings.shape)
print("Размер queries_df:", queries_df.shape)
print("Размер query_embeddings:", query_embeddings.shape)

if len(df) != len(anime_embeddings):
    raise ValueError("Количество строк df не совпадает с количеством anime_embeddings.")

if len(queries_df) != len(query_embeddings):
    raise ValueError("Количество запросов не совпадает с количеством query_embeddings.")


# Дополнительный safety-фильтр.
# Если эти жанры уже удалены заранее, размер не изменится.
banned_genres = ["Hentai", "Erotica", "Ecchi"]

safe_mask = ~df["Genres"].astype(str).str.contains(
    "|".join(banned_genres),
    case=False,
    na=False
)

df = df[safe_mask].reset_index(drop=True)
anime_embeddings = anime_embeddings[safe_mask.values]

print("Размер после safety-фильтра:", df.shape)
print("Размер anime_embeddings после safety-фильтра:", anime_embeddings.shape)

print("Данные успешно загружены. Модель не используется.")
display(queries_df)


## Основные функции

Ниже собраны функции проекта: cosine similarity, фильтры, поиск, ranking и красивый вывод.  
Все работает через `query_id`, то есть через заранее подготовленные embeddings запросов.


In [ ]:
# ============================================
# УРОК 1–4. Финальные функции проекта
# ============================================

PROJECT_NAME = "Anime AI Finder"


def cosine_scores(query_embedding, embeddings_matrix):
    """Считает cosine similarity между одним query embedding и матрицей anime embeddings."""
    query_embedding = np.asarray(query_embedding)
    embeddings_matrix = np.asarray(embeddings_matrix)

    query_norm = np.linalg.norm(query_embedding)
    matrix_norms = np.linalg.norm(embeddings_matrix, axis=1)

    return embeddings_matrix @ query_embedding / (matrix_norms * query_norm + 1e-10)


def get_query(query_id):
    """Возвращает текст запроса и его готовый embedding по query_id."""
    query_id = int(query_id)

    if query_id not in queries_df["query_id"].values:
        available_ids = queries_df["query_id"].tolist()
        raise ValueError(f"query_id={query_id} не найден. Доступные query_id: {available_ids}")

    row_position = queries_df.index[queries_df["query_id"] == query_id][0]

    query_text = queries_df.loc[row_position, "query"]
    query_embedding = query_embeddings[row_position]

    return query_text, query_embedding


def normalize_text(value):
    return str(value).strip().lower()


def split_genres(genres):
    return [
        genre.strip().lower()
        for genre in str(genres).split(",")
        if genre.strip()
    ]


def make_genre_mask(dataframe, genre_filter=None, genre_mode="any"):
    if genre_filter is None:
        return pd.Series(True, index=dataframe.index)

    if isinstance(genre_filter, str):
        genre_filter = [genre_filter]

    target_genres = [normalize_text(g) for g in genre_filter]

    def check(row_genres):
        current_genres = split_genres(row_genres)

        if genre_mode == "all":
            return all(g in current_genres for g in target_genres)

        return any(g in current_genres for g in target_genres)

    return dataframe["Genres"].apply(check)


def make_type_mask(dataframe, anime_type=None):
    if anime_type is None:
        return pd.Series(True, index=dataframe.index)

    return dataframe["Type"].astype(str).str.lower() == str(anime_type).lower()


def search_by_query_id(
    query_id,
    top_k=5,
    min_score=0,
    anime_type=None,
    genre_filter=None,
    genre_mode="any",
    alpha=0.8
):
    """
    Улучшенный поиск:
    1. берет заранее рассчитанный embedding запроса;
    2. считает similarity с anime embeddings;
    3. применяет фильтры;
    4. считает final_score;
    5. возвращает top-k рекомендаций.
    """
    query_text, query_embedding = get_query(query_id)

    similarities = cosine_scores(query_embedding, anime_embeddings)

    candidates = df.copy()
    candidates["similarity"] = similarities

    candidates = candidates[candidates["Score"] >= min_score]

    type_mask = make_type_mask(candidates, anime_type)
    candidates = candidates[type_mask]

    genre_mask = make_genre_mask(candidates, genre_filter, genre_mode)
    candidates = candidates[genre_mask]

    if len(candidates) == 0:
        return pd.DataFrame(columns=[
            "Name", "Genres", "Score", "Type", "Episodes",
            "similarity", "similarity_norm", "score_norm", "final_score", "Synopsis"
        ])

    sim_min = candidates["similarity"].min()
    sim_max = candidates["similarity"].max()

    if sim_max == sim_min:
        candidates["similarity_norm"] = 1.0
    else:
        candidates["similarity_norm"] = (
            candidates["similarity"] - sim_min
        ) / (sim_max - sim_min)

    candidates["score_norm"] = candidates["Score"].clip(0, 10) / 10

    candidates["final_score"] = (
        alpha * candidates["similarity_norm"] +
        (1 - alpha) * candidates["score_norm"]
    )

    candidates = candidates.sort_values(by="final_score", ascending=False)

    return candidates[[
        "Name", "Genres", "Score", "Type", "Episodes",
        "similarity", "similarity_norm", "score_norm", "final_score", "Synopsis"
    ]].head(top_k)


def show_recommendations_by_query_id(
    query_id,
    top_k=5,
    min_score=7.0,
    anime_type=None,
    genre_filter=None,
    genre_mode="any",
    alpha=0.8
):
    """Показывает рекомендации в красивом HTML-формате."""
    query_text, _ = get_query(query_id)

    results = search_by_query_id(
        query_id=query_id,
        top_k=top_k,
        min_score=min_score,
        anime_type=anime_type,
        genre_filter=genre_filter,
        genre_mode=genre_mode,
        alpha=alpha
    )

    if len(results) == 0:
        display(HTML("""
        <div style="font-family:Arial; padding:16px; border:1px solid #ddd; border-radius:12px;">
            <b>По заданным параметрам ничего не найдено. Попробуйте ослабить фильтры.</b>
        </div>
        """))
        return results

    html = f"""
    <div style="font-family:Arial; max-width:900px;">
        <h2>{PROJECT_NAME}</h2>
        <p><b>Выбранный запрос:</b> {query_text}</p>
        <p>
            <b>Параметры:</b>
            top_k={top_k},
            min_score={min_score},
            anime_type={anime_type},
            genre_filter={genre_filter},
            genre_mode={genre_mode},
            alpha={alpha}
        </p>
        <hr>
    """

    for i, (_, row) in enumerate(results.iterrows(), start=1):
        synopsis = str(row["Synopsis"])

        if len(synopsis) > 450:
            synopsis = synopsis[:450] + "..."

        html += f"""
        <div style="
            border:1px solid #ddd;
            border-radius:14px;
            padding:16px;
            margin-bottom:14px;
            background:#fafafa;
        ">
            <h3 style="margin-bottom:6px;">{i}. {row["Name"]}</h3>
            <p><b>Жанры:</b> {row["Genres"]}</p>
            <p>
                <b>Рейтинг:</b> {row["Score"]} |
                <b>Тип:</b> {row["Type"]} |
                <b>Эпизоды:</b> {row["Episodes"]}
            </p>
            <p>
                <b>Similarity:</b> {row["similarity"]:.3f} |
                <b>Final score:</b> {row["final_score"]:.3f}
            </p>
            <p>{synopsis}</p>
        </div>
        """

    html += "</div>"
    display(HTML(html))

    return results


def precision_at_k(labels, k=None):
    """Precision@K для ручной оценки качества."""
    if k is None:
        k = len(labels)

    labels = labels[:k]

    if len(labels) == 0:
        return 0

    return sum(labels) / len(labels)


print("Функции проекта готовы.")


## Финальный запуск проекта

Выбери `query_id` из таблицы доступных запросов и запусти рекомендацию.


In [ ]:
# ============================================
# УРОК 4. Финальная демонстрация
# ============================================

display(queries_df)

# Выбери query_id из таблицы выше
QUERY_ID = 0

results = show_recommendations_by_query_id(
    query_id=QUERY_ID,
    top_k=5,
    min_score=7.0,
    anime_type=None,
    genre_filter=None,
    genre_mode="any",
    alpha=0.8
)


## Мини-оценка качества

Ученик может вручную поставить `1` или `0` для первых 5 рекомендаций и получить `Precision@5`.

- `1` — рекомендация подходит под запрос;
- `0` — рекомендация не подходит.


In [ ]:
# ============================================
# УРОК 3. Мини-оценка качества через Precision@5
# ============================================

QUERY_ID = 0

eval_results = search_by_query_id(
    query_id=QUERY_ID,
    top_k=5,
    min_score=7.0,
    anime_type=None,
    genre_filter=None,
    alpha=0.8
)

display(eval_results[[
    "Name", "Genres", "Score", "similarity", "final_score", "Synopsis"
]])

# Поменяй значения после просмотра таблицы:
manual_labels = [1, 1, 1, 1, 1]

print("Precision@5:", precision_at_k(manual_labels, k=5))


## Сохранение результата

Можно сохранить последнюю выдачу в CSV-файл.


In [ ]:
# ============================================
# УРОК 4. Сохранение последней выдачи
# ============================================

if "results" in globals() and isinstance(results, pd.DataFrame):
    results.to_csv("anime_ai_finder_results.csv", index=False)
    print("Файл anime_ai_finder_results.csv сохранен.")
else:
    print("Сначала запусти финальную демонстрацию, чтобы появилась переменная results.")


## Итог

В этом проекте собрана финальная версия рекомендательной системы:

- данные и embeddings загружаются из Kaggle Dataset;
- модель не используется;
- интернет не нужен;
- пользователь выбирает готовый запрос по `query_id`;
- система считает similarity, применяет фильтры и показывает top-k рекомендаций.

Главная схема проекта:

```text
готовый query embedding
        ↓
cosine similarity с anime embeddings
        ↓
фильтры
        ↓
final_score
        ↓
top-k recommendations
```


In [ ]:
# ============================================
# ФИНАЛЬНАЯ ДЕМОНСТРАЦИЯ ПРОЕКТА
# Выбор запроса из готового списка
# ============================================

from IPython.display import display, HTML

# ============================================
# 1. Показываем все доступные запросы
# ============================================

def show_query_list():
    html = """
    <div style="font-family:Arial; max-width:900px;">
        <h2>Anime AI Finder</h2>
        <p><b>Выберите номер запроса из списка:</b></p>
        <table style="border-collapse:collapse; width:100%;">
            <tr>
                <th style="border:1px solid #ddd; padding:8px; text-align:left;">ID</th>
                <th style="border:1px solid #ddd; padding:8px; text-align:left;">Запрос</th>
            </tr>
    """

    for _, row in queries_df.sort_values("query_id").iterrows():
        html += f"""
            <tr>
                <td style="border:1px solid #ddd; padding:8px;">{row["query_id"]}</td>
                <td style="border:1px solid #ddd; padding:8px;">{row["query"]}</td>
            </tr>
        """

    html += """
        </table>
    </div>
    """

    display(HTML(html))


show_query_list()

In [ ]:
# ============================================
# 2. Выбор запроса и запуск рекомендаций
# ============================================

# Выбери номер запроса из таблицы выше
SELECTED_QUERY_ID = 19

# Настройки рекомендации
TOP_K = 5
MIN_SCORE = 7.0
ANIME_TYPE = None
GENRE_FILTER = None
GENRE_MODE = "any"
ALPHA = 0.8

show_recommendations_by_query_id(
    query_id=SELECTED_QUERY_ID,
    top_k=TOP_K,
    min_score=MIN_SCORE,
    anime_type=ANIME_TYPE,
    genre_filter=GENRE_FILTER,
    genre_mode=GENRE_MODE,
    alpha=ALPHA
)